# 원하는 포즈로 이미지 만들기 — FLUX.2-klein 4B (GGUF) + DWPose, Colab 무료 T4

**모델 사진 한 장**(누구를)과 **포즈 사진 한 장**(어떤 자세로)을 넣으면,
모델 사진 속 인물이 포즈 사진의 자세를 취한 이미지를 만드는 도구입니다.

출발점은 교안의 *FLUX.2-klein 4B로 마음껏 이미지 만들기* Colab 노트(ComfyUI + GGUF)이고, 여기에 포즈 조건을 얹었습니다.

## 어떻게 포즈를 넣나 (A안: 참조 이미지 2장)

FLUX.2 에는 BFL 공식 ControlNet 이 없습니다(2026-07 기준, 교안). 대신 FLUX.2-klein 은 **참조 이미지를 여러 장 받아 고치는 기능(multi-reference editing)** 이 모델 안에 있습니다. 이걸 씁니다.

```
포즈 사진 ──DWPose──▶ 뼈대(스틱맨) 이미지 ──┐
                                          ├─▶ FLUX.2-klein 4B (참조 2장 + 프롬프트) ─▶ 결과
모델 사진 ────────────────────────────────┘
   참조 1 = 누구를 (얼굴·머리·옷)      참조 2 = 어떤 자세로 (관절 위치)
```

- 포즈 사진을 **그대로** 넣지 않고 **뼈대만** 뽑아 넣는 이유: 원본 사진을 넣으면 그 사람의 옷·배경까지 섞여 들어올 수 있기 때문입니다. (실험 3에서 직접 비교합니다)
- 한계: ControlNet 처럼 관절을 강제로 고정하는 회로가 아니라 "참고하라"는 조건이라 자세가 완벽히 같지 않을 수 있습니다.

## 모델 구성과 라이선스

| 역할 | 파일 | 크기 | 라이선스 |
|---|---|---|---|
| 디퓨전 모델 | `flux-2-klein-4b-Q4_K_M.gguf` (unsloth) | 2.6GB | Apache 2.0 |
| 텍스트 인코더 | `Qwen3-4B-Q4_K_M.gguf` (unsloth) — 기본값 | 2.5GB | Apache 2.0 |
| (대안) 텍스트 인코더 | `qwen_3_4b.safetensors` (Comfy-Org, bf16) | 8.0GB | Apache 2.0 |
| VAE | `flux2-vae.safetensors` | 0.34GB | Apache 2.0 |
| 포즈 추출 | DWPose (comfyui_controlnet_aux 가 자동 다운로드) | 약 0.35GB | Apache 2.0 |

## 무료 Colab 에 맞춘 크기·메모리 전략

| 항목 | 교안 노트 | 이 노트 | 이유 |
|---|---|---|---|
| 출력 해상도 | 1024×1024 | **약 0.4MP** (포즈 사진 비율 유지, 예 발레 포즈 → 592×688) | 픽셀 수 약 40% → 계산량·VRAM 감소 |
| 참조 이미지 | 없음 | **0.25MP 로 축소** 후 입력 | 참조 이미지도 토큰으로 더해져 메모리를 씀 |
| 텍스트 인코더 | bf16 8GB | **GGUF Q4 2.5GB** | Colab 시스템 RAM(약 12.7GB)에서 가장 큰 부담 |
| 가이던스(cfg) | 4.0 | **1.0** | FLUX.2-klein 4B 증류판 공식 권장값(ComfyUI 공식 워크플로도 1). 실험 4에서 비교 |
| 디스크 | 약 11GB | **약 5.8GB** | 위 변경의 합 |

## 결과 파일 이름 (samples/)

| 참조 포즈 | 결과 | 모델 |
|---|---|---|
| `pose_01.png` (발레, `docs/pose/fmail-ballet.png`) | `output_01.png` | 남자 모델 `mail-model1.png` |
| `pose_02.png` (댄스, `docs/pose/fmail-dance.png`) | `output_02.png` | 여자 모델 `fmail-model1.png` |

같은 번호끼리 짝입니다. 포즈 사진과 성별이 다른 모델을 일부러 골라 "자세만 옮겨졌는지"가 드러나게 했습니다.
조건 실험 결과는 `exp1_…`, `exp2_…` 처럼 실험 번호로 시작하는 이름으로 따로 저장합니다.

## 실행 순서
1. 메뉴 **[런타임 → 런타임 유형 변경 → T4 GPU]** 후 저장
2. 아래 셀을 **위에서부터 차례로** 실행
3. 마지막 셀에서 `samples.zip` 을 내려받아 저장소의 `samples/` 에 넣습니다.

## [1] GPU 확인

**이 셀이 하는 일:** 런타임에 GPU 가 붙었는지, 어떤 GPU 이고 VRAM 이 얼마인지, 시스템 RAM 과 디스크가 얼마나 남았는지 확인합니다.
무료 티어는 보통 Tesla T4(VRAM 약 15GB)가 붙지만 보장되지 않습니다. GPU 가 없으면 런타임 유형을 바꾸고 다시 실행하세요.

In [ ]:
# [1] GPU·RAM·디스크 확인
import torch, shutil, os

if not torch.cuda.is_available():
    raise SystemExit("GPU가 없습니다. [런타임 → 런타임 유형 변경 → T4 GPU] 후 다시 실행하세요.")

p = torch.cuda.get_device_properties(0)
print(f"GPU  : {p.name} / VRAM {p.total_memory/1024**3:.1f}GB")
print(f"RAM  : {os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')/1024**3:.1f}GB")
print(f"디스크 여유: {shutil.disk_usage('/content').free/1024**3:.1f}GB")

## [2] ComfyUI + GGUF 노드 + 포즈 추출 노드 설치

**이 셀이 하는 일:** 세 가지를 설치합니다.
- **ComfyUI** — 노드를 이어 이미지 생성 파이프라인을 만드는 도구. 여기서는 화면 없이 **API 서버**로만 씁니다.
- **ComfyUI-GGUF** — 양자화(GGUF) 모델을 읽는 플러그인. 디퓨전 모델과 텍스트 인코더를 둘 다 GGUF 로 읽습니다.
- **comfyui_controlnet_aux** — 사진에서 사람 관절을 뽑는 **DWPose** 노드가 들어 있는 플러그인.

이미 설치돼 있으면 건너뜁니다. 약 2~4분 걸립니다.

In [ ]:
# [2] ComfyUI 본체 + 플러그인 2개 설치
import os
%cd /content
if not os.path.isdir("/content/ComfyUI"):
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt

%cd /content/ComfyUI/custom_nodes
if not os.path.isdir("ComfyUI-GGUF"):
    !git clone -q https://github.com/city96/ComfyUI-GGUF.git          # GGUF 로더
if not os.path.isdir("comfyui_controlnet_aux"):
    !git clone -q https://github.com/Fannovel16/comfyui_controlnet_aux.git  # DWPose 포함
!pip install -q gguf
!pip install -q -r comfyui_controlnet_aux/requirements.txt
%cd /content/ComfyUI
print("\n완료: ComfyUI + GGUF + controlnet_aux 설치")

## [3] 설정값 정하기와 모델 다운로드

**이 셀이 하는 일:** 이 노트 전체에서 쓰는 설정값을 한곳에 모으고, 모델 파일 3개를 ComfyUI 표준 폴더에 내려받습니다.

- `TEXT_ENCODER = "gguf"` (기본) — Qwen3-4B Q4 2.5GB. RAM 이 넉넉하거나 GGUF 인코더에서 오류가 나면 `"bf16"` 으로 바꿉니다(8GB).
- `TARGET_MP` — 출력 이미지 크기(메가픽셀). 메모리가 모자라면 0.3 으로 낮춥니다.
- `REF_MP` — 참조 이미지를 줄일 크기(메가픽셀).
- `STEPS`, `CFG`, `SEED` — 생성 설정. 증류 모델 권장값은 4스텝·cfg 1.0 입니다. **시드를 고정**해야 결과 차이가 조건 때문이라고 말할 수 있습니다.

이미 받은 파일은 건너뜁니다.

In [ ]:
# [3] 설정값 + 모델 다운로드
import os
%cd /content/ComfyUI

# ---- 설정값 (여기만 고치면 됩니다) ----
TEXT_ENCODER = "gguf"   # "gguf"(2.5GB, 기본) 또는 "bf16"(8GB)
TARGET_MP    = 0.4      # 출력 크기(메가픽셀). 메모리 부족 시 0.3
REF_MP       = 0.25     # 참조 이미지 축소 크기(메가픽셀)
STEPS        = 4        # 증류 모델 권장 4스텝
CFG          = 1.0      # FLUX.2-klein 4B 증류판 권장 1.0 (교안 노트는 4.0)
SEED         = 42       # 고정 시드

UNET_FILE = "flux-2-klein-4b-Q4_K_M.gguf"
VAE_FILE  = "flux2-vae.safetensors"
TE_FILE   = "Qwen3-4B-Q4_K_M.gguf" if TEXT_ENCODER == "gguf" else "qwen_3_4b.safetensors"

FILES = {
    f"models/diffusion_models/{UNET_FILE}":
        "https://huggingface.co/unsloth/FLUX.2-klein-4B-GGUF/resolve/main/flux-2-klein-4b-Q4_K_M.gguf",
    f"models/vae/{VAE_FILE}":
        "https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/vae/flux2-vae.safetensors",
    f"models/text_encoders/{TE_FILE}": (
        "https://huggingface.co/unsloth/Qwen3-4B-GGUF/resolve/main/Qwen3-4B-Q4_K_M.gguf"
        if TEXT_ENCODER == "gguf" else
        "https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors"),
}

for path, url in FILES.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if os.path.exists(path) and os.path.getsize(path) > 1_000_000:
        print(f"이미 있음: {path} ({os.path.getsize(path)/1e9:.2f}GB)")
        continue
    print(f"다운로드: {path}")
    !wget -q --show-progress -O "{path}" "{url}"

!du -sh models/diffusion_models models/text_encoders models/vae

## [4] ComfyUI 서버 실행

**이 셀이 하는 일:** ComfyUI 를 백그라운드 서버로 띄웁니다(`http://127.0.0.1:8188`). 다음 셀들은 이 서버에 **워크플로(JSON)** 를 보내 이미지를 만듭니다.

- `LOWVRAM = True` 로 바꾸면 `--lowvram` 옵션으로 띄웁니다. 메모리 부족(OOM)으로 죽을 때 씁니다(대신 느려짐).
- 서버가 안 뜨면 로그 끝부분이 출력됩니다.

In [ ]:
# [4] ComfyUI 서버를 백그라운드로 실행
import subprocess, time, os
%cd /content/ComfyUI

LOWVRAM = False   # OOM 이 나면 True 로 바꾸고 이 셀부터 다시 실행

!pkill -f "main.py --listen" 2>/dev/null
time.sleep(2)
opts = "--lowvram" if LOWVRAM else ""
os.system(f"nohup python main.py --listen 127.0.0.1 --port 8188 {opts} > /content/comfyui.log 2>&1 &")

SERVER = "http://127.0.0.1:8188"
for _ in range(90):
    time.sleep(1)
    code = subprocess.run(f"curl -s -o /dev/null -w '%{{http_code}}' {SERVER}/", shell=True,
                          capture_output=True, text=True).stdout
    if code == "200":
        print("완료: ComfyUI 서버 준비", SERVER)
        break
else:
    print("서버가 안 뜹니다. 로그 끝부분:")
    print(open("/content/comfyui.log").read()[-1500:])

# 필요한 노드가 모두 등록됐는지 확인
import requests
need = ["UnetLoaderGGUF", "CLIPLoaderGGUF", "DWPreprocessor", "ReferenceLatent",
        "Flux2Scheduler", "EmptyFlux2LatentImage", "CFGGuider", "SamplerCustomAdvanced"]
info = requests.get(f"{SERVER}/object_info", timeout=60).json()
missing = [n for n in need if n not in info]
print("노드 확인:", "모두 있음" if not missing else f"없음 → {missing}")

## [5] 모델 사진·포즈 사진 준비

**이 셀이 하는 일:** 생성에 쓸 사진을 ComfyUI 입력 폴더(`ComfyUI/input/`)에 넣고 화면에 보여 줍니다.

- **모델 사진** (`docs/model/`) — 포즈를 취하게 할 인물. 얼굴·머리·옷을 여기서 가져옵니다.
- **포즈 사진** (`docs/pose/`) — 따라 할 자세. 여기서 관절만 뽑아 씁니다.

사진은 **파일로 업로드**합니다. 저장소의 `docs/model/`, `docs/pose/` 사진을 내려받아 두고, 셀을 실행하면 업로드 창이 두 번 뜹니다.
1. 첫 번째 창: 모델 사진 (`fmail-model1.png`, `mail-model1.png`)
2. 두 번째 창: 포즈 사진 (`fmail-ballet.png`, `fmail-dance.png`)

(선택) `REPO_URL` 에 저장소 주소를 적으면 업로드 대신 `git clone` 으로 가져옵니다. 기본은 빈 칸(업로드)입니다.

파일 이름이 위와 같으면 [8]에서 정한 짝(발레=남자 모델, 댄스=여자 모델)으로 자동으로 맞춰집니다.

> 이 저장소의 사진 4장은 모두 AI 로 생성한 이미지입니다. 실존 인물 사진을 쓸 때는 초상권을 확인하세요.

In [ ]:
# [5] 사진 가져오기 → ComfyUI/input/ 에 복사
import os, glob, shutil
from PIL import Image
import matplotlib.pyplot as plt

REPO_URL = ""   # 기본: 빈 칸 → 업로드 창. (선택) 저장소 주소를 적으면 git clone
INPUT_DIR = "/content/ComfyUI/input"
os.makedirs(INPUT_DIR, exist_ok=True)

MODEL_IMGS, POSE_IMGS = [], []
if REPO_URL:
    !rm -rf /content/repo && git clone -q {REPO_URL} /content/repo
    for src_dir, bucket in [("/content/repo/docs/model", MODEL_IMGS), ("/content/repo/docs/pose", POSE_IMGS)]:
        for f in sorted(glob.glob(f"{src_dir}/*.png") + glob.glob(f"{src_dir}/*.jpg")):
            shutil.copy(f, INPUT_DIR); bucket.append(os.path.basename(f))
else:
    from google.colab import files
    for label, bucket in [("모델 사진", MODEL_IMGS), ("포즈 사진", POSE_IMGS)]:
        print(f"▶ {label}을 고르세요")
        for name, data in files.upload().items():
            open(f"{INPUT_DIR}/{name}", "wb").write(data); bucket.append(name)

print("모델 사진:", MODEL_IMGS)
print("포즈 사진:", POSE_IMGS)

def show(paths, titles, h=4):
    # 이미지 여러 장을 한 줄로 보여 주는 도우미
    fig, axs = plt.subplots(1, len(paths), figsize=(h*len(paths), h))
    axs = [axs] if len(paths) == 1 else axs
    for ax, pth, t in zip(axs, paths, titles):
        ax.imshow(Image.open(pth)); ax.set_title(t, fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()

show([f"{INPUT_DIR}/{n}" for n in MODEL_IMGS + POSE_IMGS],
     [f"model: {n}" for n in MODEL_IMGS] + [f"pose: {n}" for n in POSE_IMGS])
for n in MODEL_IMGS + POSE_IMGS:
    print(n, Image.open(f"{INPUT_DIR}/{n}").size)

## [6] 포즈 추출 (DWPose)

**이 셀이 하는 일:** 포즈 사진마다 DWPose 로 관절을 뽑아 **뼈대 이미지**를 만들고 저장합니다.

- 처음 실행할 때 DWPose 모델(약 0.35GB)을 자동으로 내려받아 조금 걸립니다.
- 몸·손·얼굴 관절을 모두 뽑습니다(`detect_hand/body/face = enable`).
- 여기서 뼈대가 이상하게 나오면(팔이 빠짐, 사람이 둘로 잡힘) 그 포즈 사진으로는 결과도 어긋납니다. **생성 전에 눈으로 확인**하는 단계입니다.

ComfyUI 에 보내는 **워크플로 JSON** 은 노드 번호를 키로, `class_type`(노드 종류)과 `inputs`(설정값·다른 노드 출력 연결)를 값으로 갖는 딕셔너리입니다. `["20", 0]` 은 "20번 노드의 0번째 출력"이라는 뜻입니다.

In [ ]:
# [6] 공통 도우미: 워크플로 전송 → 완료 대기 → 결과 이미지 받기
import requests, time, json, urllib.parse, uuid

def run_workflow(workflow, timeout_s=600):
    # ComfyUI 에 워크플로를 보내고, 끝나면 {노드번호: [이미지 바이트, ...]} 를 돌려준다
    r = requests.post(f"{SERVER}/prompt", json={"prompt": workflow, "client_id": str(uuid.uuid4())}, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"요청 실패 {r.status_code}: {r.text[:800]}")
    pid, t0 = r.json()["prompt_id"], time.time()
    while time.time() - t0 < timeout_s:
        time.sleep(1.5)
        h = requests.get(f"{SERVER}/history/{pid}", timeout=30).json()
        if pid not in h:
            continue
        entry = h[pid]
        if entry.get("status", {}).get("status_str") == "error":
            msgs = json.dumps(entry["status"].get("messages", []), ensure_ascii=False)
            raise RuntimeError("ComfyUI 실행 오류: " + msgs[-1500:])
        result = {}
        for node_id, out in entry.get("outputs", {}).items():
            for img in out.get("images", []):
                q = urllib.parse.urlencode({"filename": img["filename"], "subfolder": img.get("subfolder", ""),
                                            "type": img.get("type", "output")})
                result.setdefault(node_id, []).append(requests.get(f"{SERVER}/view?{q}", timeout=60).content)
        return result, round(time.time() - t0, 1)
    raise TimeoutError("시간 초과")

DWPOSE_INPUTS = {"detect_hand": "enable", "detect_body": "enable", "detect_face": "enable",
                 "resolution": 512, "bbox_detector": "yolox_l.onnx",
                 "pose_estimator": "dw-ll_ucoco_384_bs5.torchscript.pt",
                 "scale_stick_for_xinsr_cn": "disable"}

# 포즈 사진마다 뼈대만 뽑아서 저장
os.makedirs("/content/samples", exist_ok=True)
SKELETONS = {}
for i, pose in enumerate(POSE_IMGS, 1):
    wf = {
        "20": {"class_type": "LoadImage", "inputs": {"image": pose}},
        "21": {"class_type": "DWPreprocessor", "inputs": {"image": ["20", 0], **DWPOSE_INPUTS}},
        "24": {"class_type": "SaveImage", "inputs": {"images": ["21", 0], "filename_prefix": "skeleton"}},
    }
    out, sec = run_workflow(wf)
    skel_path = f"/content/samples/pose_{i:02d}_skeleton.png"
    open(skel_path, "wb").write(out["24"][0])
    shutil.copy(f"{INPUT_DIR}/{pose}", f"/content/samples/pose_{i:02d}.png")
    SKELETONS[pose] = skel_path
    print(f"pose_{i:02d}: {pose} → 뼈대 추출 {sec}초")

show([f"{INPUT_DIR}/{p}" for p in POSE_IMGS] + list(SKELETONS.values()),
     POSE_IMGS + [f"skeleton of {p}" for p in POSE_IMGS])

## [7] 포즈 조건 생성 함수 만들기

**이 셀이 하는 일:** "모델 사진 + 포즈 사진 + 프롬프트 → 결과 이미지" 를 한 번에 하는 함수 `make_pose_image()` 를 정의합니다. (아직 생성하지 않습니다)

워크플로 구조는 ComfyUI 공식 템플릿 *Image Edit (FLUX.2 Klein 4B Distilled)* 의 참조 2장 구성을 따르고, 로더만 GGUF 로 바꿨습니다.

| 단계 | 노드 | 하는 일 |
|---|---|---|
| 모델 로드 | `UnetLoaderGGUF`, `CLIPLoaderGGUF`(또는 `CLIPLoader`), `VAELoader` | 디퓨전 모델·텍스트 인코더·VAE |
| 참조 1 | `LoadImage` → `ImageScaleToTotalPixels`(0.25MP) → `VAEEncode` | 모델 사진을 줄여 잠재 공간으로 |
| 참조 2 | `LoadImage` → `DWPreprocessor` → 축소 → `VAEEncode` | 포즈 사진에서 뼈대를 뽑아 잠재 공간으로 |
| 조건 | `CLIPTextEncode` → `ReferenceLatent` ×2 | 프롬프트에 참조 2장을 순서대로 붙임 (이미지 1, 이미지 2) |
| 생성 | `Flux2Scheduler`(4스텝) + `CFGGuider`(cfg 1.0) + `SamplerCustomAdvanced` | 빈 캔버스에서 잡음을 걷어 냄 |
| 출력 | `VAEDecode` → `SaveImage` | 픽셀 이미지로 펴서 저장 |

- 출력 크기는 **포즈 사진의 가로세로 비율**을 따르고, 넓이는 `TARGET_MP` 로 맞춥니다(16의 배수). 뼈대와 캔버스 비율이 같아야 자세가 덜 찌그러집니다.
- `pose_mode="photo"` 로 부르면 뼈대 대신 **원본 포즈 사진**을 그대로 참조로 넣습니다(실험 3 비교용).

In [ ]:
# [7] 포즈 조건 생성 함수
def out_size(pose_file, target_mp=None):
    # 포즈 사진 비율을 유지하면서 넓이를 target_mp 메가픽셀로 맞춘 (가로, 세로). 16의 배수
    target_mp = target_mp or TARGET_MP
    w, h = Image.open(f"{INPUT_DIR}/{pose_file}").size
    s = (target_mp * 1e6 / (w * h)) ** 0.5
    return max(256, round(w * s / 16) * 16), max(256, round(h * s / 16) * 16)

def build_workflow(model_file, pose_file, prompt, seed=None, cfg=None, steps=None, pose_mode="skeleton"):
    W, H = out_size(pose_file)
    te_loader = ({"class_type": "CLIPLoaderGGUF", "inputs": {"clip_name": TE_FILE, "type": "flux2"}}
                 if TE_FILE.endswith(".gguf") else
                 {"class_type": "CLIPLoader", "inputs": {"clip_name": TE_FILE, "type": "flux2"}})
    pose_src = ["21", 0] if pose_mode == "skeleton" else ["20", 0]
    wf = {
        # 모델 로드
        "1":  {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": UNET_FILE}},
        "2":  te_loader,
        "3":  {"class_type": "VAELoader", "inputs": {"vae_name": VAE_FILE}},
        # 참조 1: 모델 사진 (누구를)
        "10": {"class_type": "LoadImage", "inputs": {"image": model_file}},
        "11": {"class_type": "ImageScaleToTotalPixels", "inputs": {"image": ["10", 0], "upscale_method": "lanczos",
                                                                  "megapixels": REF_MP, "resolution_steps": 1}},
        "12": {"class_type": "VAEEncode", "inputs": {"pixels": ["11", 0], "vae": ["3", 0]}},
        # 참조 2: 포즈 (어떤 자세로)
        "20": {"class_type": "LoadImage", "inputs": {"image": pose_file}},
        "22": {"class_type": "ImageScaleToTotalPixels", "inputs": {"image": pose_src, "upscale_method": "lanczos",
                                                                  "megapixels": REF_MP, "resolution_steps": 1}},
        "23": {"class_type": "VAEEncode", "inputs": {"pixels": ["22", 0], "vae": ["3", 0]}},
        # 조건: 프롬프트 + 참조 2장 (빈 조건 쪽에도 같은 참조를 붙임 — 공식 템플릿과 동일)
        "30": {"class_type": "CLIPTextEncode", "inputs": {"text": prompt, "clip": ["2", 0]}},
        "31": {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": ["30", 0]}},
        "32": {"class_type": "ReferenceLatent", "inputs": {"conditioning": ["30", 0], "latent": ["12", 0]}},
        "33": {"class_type": "ReferenceLatent", "inputs": {"conditioning": ["32", 0], "latent": ["23", 0]}},
        "34": {"class_type": "ReferenceLatent", "inputs": {"conditioning": ["31", 0], "latent": ["12", 0]}},
        "35": {"class_type": "ReferenceLatent", "inputs": {"conditioning": ["34", 0], "latent": ["23", 0]}},
        # 생성
        "40": {"class_type": "CFGGuider", "inputs": {"model": ["1", 0], "positive": ["33", 0],
                                                      "negative": ["35", 0], "cfg": cfg if cfg is not None else CFG}},
        "41": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "euler"}},
        "42": {"class_type": "Flux2Scheduler", "inputs": {"steps": steps or STEPS, "width": W, "height": H}},
        "43": {"class_type": "RandomNoise", "inputs": {"noise_seed": seed if seed is not None else SEED}},
        "44": {"class_type": "EmptyFlux2LatentImage", "inputs": {"width": W, "height": H, "batch_size": 1}},
        "45": {"class_type": "SamplerCustomAdvanced", "inputs": {"noise": ["43", 0], "guider": ["40", 0],
                    "sampler": ["41", 0], "sigmas": ["42", 0], "latent_image": ["44", 0]}},
        # 출력
        "46": {"class_type": "VAEDecode", "inputs": {"samples": ["45", 0], "vae": ["3", 0]}},
        "47": {"class_type": "SaveImage", "inputs": {"images": ["46", 0], "filename_prefix": "pose_out"}},
    }
    if pose_mode == "skeleton":
        wf["21"] = {"class_type": "DWPreprocessor", "inputs": {"image": ["20", 0], **DWPOSE_INPUTS}}
    return wf, (W, H)

LOG = []   # 실험 기록 (prompts.md 로 옮길 내용)

def make_pose_image(out_id, model_file, pose_file, prompt, seed=None, cfg=None, steps=None,
                    pose_mode="skeleton", note=""):
    # 생성 → /content/samples/{out_id}.png 저장 → 기록 남기기
    wf, (W, H) = build_workflow(model_file, pose_file, prompt, seed, cfg, steps, pose_mode)
    out, sec = run_workflow(wf)
    path = f"/content/samples/{out_id}.png"
    open(path, "wb").write(out["47"][0])
    rec = {"id": out_id, "model": model_file, "pose": pose_file, "pose_mode": pose_mode, "prompt": prompt,
           "seed": seed if seed is not None else SEED, "cfg": cfg if cfg is not None else CFG,
           "steps": steps or STEPS, "size": f"{W}x{H}", "unet": UNET_FILE, "text_encoder": TE_FILE,
           "seconds": sec, "note": note}
    json.dump(rec, open(f"/content/samples/{out_id}.json", "w"), ensure_ascii=False, indent=1)
    LOG.append(rec)
    print(f"{out_id}: {W}x{H}, {sec}초")
    return path

print("함수 준비 완료: make_pose_image(out_id, model_file, pose_file, prompt, ...)")

## [8] 도구 동작 확인 — output_01, output_02 만들기

**이 셀이 하는 일:** 포즈마다 결과를 한 장씩 만들어 `samples/` 의 짝 이름으로 저장하고, 참조 2장과 결과를 나란히 보여 줍니다.

| 결과 | 모델 사진 (이미지 1) | 포즈 사진 → 뼈대 (이미지 2) |
|---|---|---|
| `output_01.png` | 남자 모델 `mail-model1.png` | `pose_01` 발레 `fmail-ballet.png` |
| `output_02.png` | 여자 모델 `fmail-model1.png` | `pose_02` 댄스 `fmail-dance.png` |

업로드한 파일 이름이 다르면 `PAIRS` 의 파일 이름을 고쳐 주세요.

프롬프트는 "이미지 1의 사람이 이미지 2 뼈대의 자세를 취한다"를 영어 문장으로 적었습니다. 참조 이미지의 순서(이미지 1 = 모델, 이미지 2 = 뼈대)를 문장에서도 그대로 부릅니다.

- 첫 실행은 모델을 GPU 에 올리느라 오래 걸리고, 그다음부터 빨라집니다.
- 여기서 **메모리 부족(OOM)** 이 나면: [3]의 `TARGET_MP` 를 0.3 으로 → [4]의 `LOWVRAM=True` → 순서대로 시도합니다.

In [ ]:
# [8] 도구 동작 확인: output_01, output_02
BASE_PROMPT = ("The person from image 1 strikes exactly the same full-body pose as the stick-figure skeleton "
               "in image 2. Keep the face, hairstyle and outfit of the person in image 1. "
               "Full-body shot, the whole figure visible from head to feet, plain light grey studio background, "
               "soft even lighting. Photorealistic fashion photography.")

MALE, FEMALE = "mail-model1.png", "fmail-model1.png"
BALLET, DANCE = "fmail-ballet.png", "fmail-dance.png"
PAIRS = [("output_01", MALE, BALLET),    # pose_01 ↔ output_01
         ("output_02", FEMALE, DANCE)]   # pose_02 ↔ output_02

for out_id, m, p in PAIRS:
    assert m in MODEL_IMGS and p in POSE_IMGS, f"업로드한 파일에 {m} 또는 {p} 가 없습니다. PAIRS 를 고치세요."
    path = make_pose_image(out_id, m, p, BASE_PROMPT, note=f"도구 동작: {m} × {p}")
    show([f"{INPUT_DIR}/{m}", SKELETONS[p], f"{INPUT_DIR}/{p}", path],
         ["image 1: model", "image 2: skeleton", "pose photo", out_id])

## [9] 조건을 바꿔 보는 실험

**이 셀이 하는 일:** 조건을 하나씩만 바꿔 가며 결과를 만들고 나란히 보여 줍니다. **시드는 모두 같게(42)** 고정해 차이가 바꾼 조건 때문이라고 말할 수 있게 했습니다.
실험 결과는 `output_XX` 와 섞이지 않게 `exp{번호}_{내용}.png` 로 저장합니다.

| 실험 | 고정 | 바꾸는 것 | 결과 파일 | 보고 싶은 것 |
|---|---|---|---|---|
| 1. 같은 포즈 · 다른 프롬프트 | 남자 모델 × pose_01(발레) | 배경·조명 / 옷 / 화풍 | `exp1_stage`, `exp1_costume`, `exp1_anime` | 프롬프트를 바꿔도 자세가 유지되는가 (기준: output_01) |
| 2. 같은 프롬프트 · 다른 포즈 | BASE 프롬프트 | 반대 짝: 여자×발레, 남자×댄스 | `exp2_female_ballet`, `exp2_male_dance` | 포즈를 바꾸면 자세가 따라 바뀌는가, 인물은 유지되는가 (output_01·02와 2×2 비교) |
| 3. 뼈대 vs 원본 사진 | 남자 모델 × pose_01, BASE | 참조 2 = 원본 포즈 사진 | `exp3_photo_ref` | 원본을 넣으면 옷·배경·얼굴이 섞여 들어오는가 |
| 4. cfg 1.0 vs 4.0 | 남자 모델 × pose_01, BASE | cfg 4.0 | `exp4_cfg4` | 교안 노트 값(4.0)이 증류 모델에서 어떻게 보이는가 |

In [ ]:
# [9-1] 실험 1: 같은 포즈 · 다른 프롬프트 (남자 모델 × 발레, 기준 = output_01)
HEAD = "The person from image 1 strikes exactly the same full-body pose as the stick-figure skeleton in image 2. "
E1 = {
    "exp1_stage":   (HEAD + "Keep the face, hairstyle and outfit of the person in image 1. Full-body shot on a theatre stage, "
                     "deep red velvet curtain behind, warm spotlight from above. Photorealistic stage photography.",
                     "실험1 환경·조명: 무대"),
    "exp1_costume": (HEAD + "Keep the face and hairstyle of the person in image 1, but dress them in a fitted white ballet "
                     "costume with white tights and ballet slippers. Full-body shot, plain light grey studio background, "
                     "soft even lighting. Photorealistic fashion photography.",
                     "실험1 옷: 발레복"),
    "exp1_anime":   (HEAD + "Keep the face, hairstyle and outfit of the person in image 1. Full-body shot, plain cream background. "
                     "Clean flat 2D anime illustration with bold outlines and soft cel shading.",
                     "실험1 스타일: 일러스트"),
}
paths = [make_pose_image(k, MALE, BALLET, pr, note=nt) for k, (pr, nt) in E1.items()]
show([SKELETONS[BALLET], "/content/samples/output_01.png"] + paths,
     ["skeleton (pose_01)", "output_01 (BASE)"] + list(E1.keys()), h=3.2)

In [ ]:
# [9-2] 실험 2: 같은 프롬프트 · 다른 포즈 — 반대 짝 2장을 만들어 output_01·02 와 2×2 로 비교
e2a = make_pose_image("exp2_female_ballet", FEMALE, BALLET, BASE_PROMPT, note="실험2: 여자 모델 × 발레")
e2b = make_pose_image("exp2_male_dance",    MALE,   DANCE,  BASE_PROMPT, note="실험2: 남자 모델 × 댄스")
show(["/content/samples/output_01.png", e2b, e2a, "/content/samples/output_02.png"],
     ["male x ballet (output_01)", "male x dance", "female x ballet", "female x dance (output_02)"], h=3.2)

In [ ]:
# [9-3] 실험 3: 뼈대 vs 원본 포즈 사진 / [9-4] 실험 4: cfg 1.0 vs 4.0  (모두 남자 모델 × 발레)
e3 = make_pose_image("exp3_photo_ref", MALE, BALLET, BASE_PROMPT, pose_mode="photo",
                     note="실험3: 참조 2 = 원본 포즈 사진")
e4 = make_pose_image("exp4_cfg4", MALE, BALLET, BASE_PROMPT, cfg=4.0,
                     note="실험4: cfg 4.0 (교안 노트 값)")
show(["/content/samples/output_01.png", e3, e4],
     ["skeleton ref, cfg 1.0 (output_01)", "photo ref (exp3)", "cfg 4.0 (exp4)"], h=3.5)

## [10] 결과 정리와 내려받기

**이 셀이 하는 일:** 모든 생성 기록을 표(마크다운)로 출력하고, `samples/` 폴더 전체(참조 포즈 사진, 뼈대, 결과, 설정 JSON)와 기록 파일을 `samples.zip` 으로 묶어 내려받습니다.
출력된 표는 저장소의 `prompts.md` 에 붙여 넣습니다.

In [ ]:
# [10] 기록 표 출력 + samples.zip 내려받기
rows = ["| 파일 | 모델 | 포즈 | 참조2 | cfg | 크기 | 시간(초) | 메모 |", "|---|---|---|---|---|---|---|---|"]
for r in LOG:
    rows.append(f"| {r['id']} | {r['model']} | {r['pose']} | {r['pose_mode']} | {r['cfg']} | {r['size']} | {r['seconds']} | {r['note']} |")
table = "\n".join(rows)
prompts = "\n\n".join(f"**{r['id']}** (seed {r['seed']})\n\n```\n{r['prompt']}\n```" for r in LOG)
open("/content/samples/experiment_log.md", "w").write(f"## 실험 기록\n\n{table}\n\n## 프롬프트\n\n{prompts}\n")
print(table)

!cd /content && rm -f samples.zip && zip -q -r samples.zip samples
from google.colab import files
files.download("/content/samples.zip")

## [11] 관찰 기록 — 무엇을 바꿔 보았고, 어떻게 달라졌는가

> 2026-11-11, 무료 Colab Tesla T4 에서 실행한 결과를 직접 보고 적었습니다. 모든 생성은 seed 42, 4스텝, 출력 약 0.4MP 입니다. "(추측)"이라고 적은 것은 확인하지 않은 해석입니다.

### 먼저: 뼈대 확인 ([6])
- 발레 사진: 튀튀에 가려 **뒤로 든 다리를 잡지 못했습니다**. 두 다리가 모두 아래로 겹친 뼈대가 나왔습니다. 팔 두 개와 몸통은 정확합니다.
- 댄스 사진: 주인공은 잘 잡혔지만, **배경의 사람이 작은 뼈대로 하나 더** 잡혔습니다.

### 도구 동작 (output_01 · output_02)
- output_01 (남자 모델 × pose_01 발레): 위로 뻗은 팔은 뼈대와 같습니다. 하지만 옆으로 뻗어야 할 팔은 주머니에 들어가 있고, 다리는 아라베스크가 아니라 서 있는 자세입니다. **자세가 일부만 옮겨졌습니다.** 얼굴·코트·회색 배경은 지시대로 나왔습니다.
- output_02 (여자 모델 × pose_02 댄스): 양팔을 옆으로 뻗고 한 다리를 옆으로 차올린 자세가 뼈대와 **대체로 같습니다.** 얼굴·재킷·가방·원피스도 유지됐습니다. 몸의 기울기나 무릎 각도는 원본보다 얌전합니다.

### 실험 1. 같은 포즈 · 다른 프롬프트 (output_01 vs exp1_*)
- 바꾼 것: 환경·조명(무대) / 옷(발레복) / 스타일(2D 일러스트)
- 자세 유지 정도: 오히려 **세 장 모두 output_01 보다 팔이 뼈대에 더 가깝습니다.** 위팔과 옆으로 뻗은 팔이 모두 맞습니다. 다리는 네 장 모두 서 있는 자세입니다(뼈대에 뒤로 든 다리가 없으므로).
- 프롬프트가 반영된 정도: 무대 커튼·스포트라이트, 흰 발레복·타이츠, 셀 셰이딩 일러스트가 모두 잘 반영됐습니다. 다만 **발레복 프롬프트에서는 남자 모델이 여성 발레리나로 바뀌었습니다.** 얼굴까지 바뀌어 참조 1(인물)이 지켜지지 않았습니다. "ballet costume"이라는 말이 모델 사진보다 강하게 작용한 것 같습니다(추측).
- 정리: 시드가 같아도 프롬프트 문장이 바뀌면 팔 위치가 달라집니다. output_01 의 "주머니 팔"은 뼈대 문제라기보다 이 시드·문장 조합의 흔들림으로 보입니다(추측).

### 실험 2. 같은 프롬프트 · 다른 포즈 (2×2: output_01, output_02, exp2_*)
- 포즈를 바꿨을 때 자세가 따라 바뀌었나: **예.** 발레 뼈대에서는 남녀 모두 "위팔 하나 + 서 있기"가, 댄스 뼈대에서는 남녀 모두 "양팔 벌리기 + 한 다리 차올리기"가 나왔습니다.
- 인물(얼굴·옷)은 유지됐나: **예.** 네 장 모두 각자의 얼굴·코트·재킷·가방이 유지됐습니다.
- 이상한 점: `exp2_male_dance` 에는 **다리가 3개**이고, 왼쪽에 **작은 사람이 하나 더** 생겼습니다. 작은 사람은 [6]에서 잡힌 배경 인물의 뼈대가 그대로 그려진 것입니다. 다리 3개는 원래 모델 사진의 서 있는 다리 두 개와 뼈대의 차올린 다리가 함께 그려진 것 같습니다(추측). 원피스를 입은 여자 모델은 다리가 가려져 이 문제가 드러나지 않았습니다(추측).
- 어느 쪽 변화가 결과를 더 크게 바꿨나 (포즈 vs 프롬프트): 자세는 **포즈(뼈대)**, 배경·옷·화풍은 **프롬프트**가 정했습니다. 실험 1의 프롬프트 변화는 팔 위치를 조금 바꿨을 뿐이지만, 실험 2의 포즈 변화는 전신 자세를 바꿨습니다.

### 실험 3. 뼈대 vs 원본 포즈 사진 (output_01 vs exp3_photo_ref)
- 원본 사진을 넣었을 때 섞여 들어온 것: **거의 없습니다.** 튀튀·붉은 커튼·발레리나 얼굴은 들어오지 않았고, 결과는 output_01 과 거의 같습니다(팔 각도만 조금 다름).
- 예상("옷·배경이 섞인다")과 달랐습니다. 이번 조건(참조 0.25MP, cfg 1.0)에서는 참조 2가 결과에 주는 힘이 약하고, 결과가 참조 1(모델 사진)과 프롬프트 쪽으로 기우는 것으로 보입니다(추측). output_01 이 발레를 끝까지 따라가지 못한 것과도 이어집니다.

### 실험 4. cfg 1.0 vs 4.0 (output_01 vs exp4_cfg4)
- 차이: cfg 4.0 은 **대비가 지나치게 강하고**, 배경이 하얗게 날아갔으며, 코트 질감이 거칠어졌습니다. 자세는 비슷하고 옆 팔이 조금 더 펴졌습니다. 시간은 12초 안팎에서 21.1초로 약 2배가 됐습니다.
- FLUX.2-klein 4B 증류판의 공식 권장값(cfg 1.0)을 뒷받침하는 결과입니다. 교안 노트의 4.0 은 이 모델에 맞지 않습니다.

### 한계와 다음에 해 볼 것
- 손·발끝처럼 작은 관절: 0.4MP 결과에서는 손가락까지 비교하지 않았습니다. 손 모양은 대체로 자연스럽습니다.
- 메모리·속도 (무료 T4): OOM 없이 기본 설정(TARGET_MP 0.4, LOWVRAM False, GGUF 인코더)으로 모두 돌았습니다. 첫 장 50.9초, 이후 장당 10~14초가 걸렸습니다.
- 다음에 해 볼 것: ① 다리가 가려지지 않은 포즈 사진 쓰기 ② 배경 인물이 없는 사진 쓰기, 또는 뼈대에서 작은 사람 지우기 ③ 시드 여러 개로 같은 조건 반복 ④ 참조 2의 해상도(REF_MP) 올리기